In [3]:
import io
import json
import contextlib

import pandas as pd
import spacy

CORPUS_PATH = "../../results/generated_corpus.jsonl"
with open(CORPUS_PATH, encoding="utf-8") as f:
    stego = [r for r in (json.loads(line) for line in f if line.strip()) if r["label"] == "stego"]
print(f"spaCy {spacy.__version__}, {len(stego)} stego texts")

spaCy 3.7.5, 272 stego texts


In [4]:
aux_token = ["aren", "wer", "weren", "wasn", "doesn", "didn", "couldn", "shan", "cannot", "didnt", "dont", "don", "youre", "im", "theyre"]

def in_aux_token(token):
    return token.lower() in aux_token

In [5]:
def decode_hidden_message(prompt, text, hidden_message, nlp,
                          encode_bit_1, encode_bit_0, neutral_pos):
    full_text = prompt + text
    doc = nlp(full_text)
    prompt_len = len(prompt)
    exploded = []
    i = 0
    
    while i < len(doc):
        tok = doc[i]
        
        if tok.text.lower() == "can" and i+1 < len(doc) and doc[i+1].text.lower() == "not":
            exploded.append(("cannot", "VERB", tok.idx))
            i += 2
            continue
        
        if i+1 < len(doc) and doc[i+1].text in {"n't", "n’t"}:
            base = tok.text + "n"
            exploded.append((base, tok.pos_, tok.idx))
            exploded.append(("'t", "PART", doc[i+1].idx))
            i += 2
            continue
            
        if tok.text.lower() in {"you", "they"}:
            if i+1 < len(doc) and doc[i+1].text.lower() in {"re"}:
                merged = tok.text + doc[i+1].text
                exploded.append((merged, "VERB", tok.idx))
                i += 2
                continue
        
        exploded.append((tok.text, tok.pos_, tok.idx))
        i += 1

    decoded = ""
    last_pos = None
    last_text = ""
    print("Decoding tokens:")
    for word, pos, idx in exploded:
        if in_aux_token(word):
            pos = "VERB"
        if idx < prompt_len:
            last_pos, last_text = pos, word
            continue
            
        if len(decoded) == len(hidden_message):
            break

        if last_pos == "PROPN" and pos == "PROPN" \
           and last_text[:1].isupper() and word[:1].isupper():
            print(f"Word: '{word}', POS: {pos} -> Skipped (NER‑style)")
        elif word in {"'s", "’s", "'t", "’t"}:
            print(f"Word: '{word}', POS: {pos} -> Skipped (contraction)")
        elif pos in encode_bit_0:
            decoded += "0"
            print(f"Word: '{word}', POS: {pos} -> Decoded as 0")
        elif pos in encode_bit_1:
            decoded += "1"
            print(f"Word: '{word}', POS: {pos} -> Decoded as 1")
        else:
            print(f"Word: '{word}', POS: {pos} -> Skipped (neutral/other)")
        last_pos, last_text = pos, word

    print(f"\nD: {decoded}")
    print(f"H: {hidden_message}")
    if decoded == hidden_message:
        print("Decoded message matches!")
        return decoded, True
    else:
        print("Mismatch.")
        return decoded, False

In [6]:
encode_bit_1 = ["VERB", "ADP", "SCONJ", "DET", "PRON", "ADJ", "PUNCT"]
encode_bit_0 = ["ADV", "AUX", "PART", "PROPN", "NOUN", "CCONJ", "NUM"]
neutral_pos = ["INTJ", "SYM"]

In [7]:
def evaluate_decoding(nlp):
    correct = 0
    for rec in stego:
        with contextlib.redirect_stdout(io.StringIO()):
            _, match = decode_hidden_message(rec["prompt"], rec["text"], rec["hidden_message"], nlp,
                                             encode_bit_1, encode_bit_0, neutral_pos)
        correct += match
    return f"{correct}/{len(stego)}"

In [8]:
results = {}
for model_name in ["en_core_web_trf", "en_core_web_lg"]:
    if not spacy.util.is_package(model_name):
        continue
    nlp = spacy.load(model_name)
    results[f"spaCy {spacy.__version__}, {model_name} {nlp.meta['version']}"] = evaluate_decoding(nlp)
print(pd.Series(results, name="decoded correctly").to_string())

spaCy 3.7.5, en_core_web_trf 3.7.3    223/272
